In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [2]:
class CertifiedWarpGatekeeper:
    """Stage 1: Pre-ViT pixel gatekeeper operating on raw image patches."""
    def __init__(self, delta=32, kappa=16):
        self.delta = delta
        self.kappa = kappa

    def get_active_mask(self, patch_F, patch_W):
        """
        patch_F: Raw pixels of current frame [B, N, C, P, P]
        patch_W: Raw pixels of warped canvas [B, N, C, P, P]
        Returns a boolean mask of patches to compute (True = keep, False = skip)
        """
        diff = patch_F - patch_W
        # Compute statistics per patch (flattening spatial/channel dims)
        diff_flat = diff.view(diff.shape[0], diff.shape[1], -1)
        s = diff_flat.max(dim=-1)[0] - diff_flat.min(dim=-1)[0]
        c = (diff_flat.max(dim=-1)[0] + diff_flat.min(dim=-1)[0]) / 2.0

        # Keep patch if it exceeds contrast threshold OR brightness shift cap
        keep_mask = (s >= self.delta) | (c.abs() > self.kappa)
        return keep_mask

def apply_2d_rope(tokens, coords, head_dim):
    """
    Differentiable 2D RoPE using continuous (x, y) coordinates.
    coords: [B, N, 2] representing (x, y) locations
    """
    B, N, H, D = tokens.shape
    freqs = torch.arange(0, head_dim // 2, 2, device=tokens.device).float()
    inv_freq = 1.0 / (10000 ** (freqs / (head_dim // 2)))

    # Compute angles for X and Y coordinates
    angles_x = torch.einsum("bn,f->bnf", coords[..., 0], inv_freq)
    angles_y = torch.einsum("bn,f->bnf", coords[..., 1], inv_freq)

    # Construct rotary embeddings
    sin_x, cos_x = angles_x.sin(), angles_x.cos()
    sin_y, cos_y = angles_y.sin(), angles_y.cos()

    # Broadcast to heads and apply (simplified rotation for demonstration)
    sin_x = sin_x.unsqueeze(2).expand(-1, -1, H, -1)
    cos_x = cos_x.unsqueeze(2).expand(-1, -1, H, -1)

    # Apply to first half of feature dim (X-axis)
    tokens_x = tokens[..., :head_dim//2] * cos_x + torch.roll(tokens[..., :head_dim//2], shifts=1, dims=-1) * sin_x
    # (Y-axis application omitted for brevity; exact same logic applies to second half)
    tokens_y = tokens[..., head_dim//2:]

    return torch.cat([tokens_x, tokens_y], dim=-1)

class WarpSteeredAttention(nn.Module):
    """Stage 2: ViT Attention block maintaining a World-Anchored KV Cache."""
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.o_proj = nn.Linear(embed_dim, embed_dim, bias=False)

        self.cache_k = None
        self.cache_v = None

    def forward(self, active_x, active_coords, warped_canvas_coords):
        B, N_act, D = active_x.shape

        q_act = self.q_proj(active_x).view(B, N_act, self.num_heads, self.head_dim)
        k_act = self.k_proj(active_x).view(B, N_act, self.num_heads, self.head_dim)
        v_act = self.v_proj(active_x).view(B, N_act, self.num_heads, self.head_dim)

        # Manage the World-Anchored KV Cache
        if self.cache_k is None:
            self.cache_k = k_act
            self.cache_v = v_act
        else:
            # In a real model, you overwrite specific index slots here.
            # We append for the simulation to represent memory growth.
            self.cache_k = torch.cat([self.cache_k, k_act], dim=1)
            self.cache_v = torch.cat([self.cache_v, v_act], dim=1)

        # Rotate active Queries by their current native coordinates
        q_rotated = apply_2d_rope(q_act, active_coords, self.head_dim)

        # Rotate ENTIRE un-rotated Key cache by the newly warped coordinates g_t(p)
        k_rotated_cache = apply_2d_rope(self.cache_k, warped_canvas_coords, self.head_dim)

        # FlashAttention Matrix Multiply
        scores = torch.einsum("bnhd,bmhd->bhnm", q_rotated, k_rotated_cache) / (self.head_dim ** 0.5)
        attn = F.softmax(scores, dim=-1)

        out = torch.einsum("bhnm,bmhd->bnhd", attn, self.cache_v)
        return self.o_proj(out.reshape(B, N_act, D))


class VideoModel(nn.Module):
    """End-to-end model pipeline connecting patch embedding, attention, and task head."""
    def __init__(self, embed_dim=128, num_heads=4):
        super().__init__()
        self.patch_embed = nn.Linear(16*16*3, embed_dim) # Assuming 16x16 RGB patches
        self.attention = WarpSteeredAttention(embed_dim, num_heads)
        self.task_head = nn.Linear(embed_dim, 10) # 10-class output prediction

    def forward(self, active_pixels, active_coords, warped_canvas_coords):
        # 1. Project raw pixels to latent vectors
        x = self.patch_embed(active_pixels)

        # 2. Process through Warp-Steered Attention
        x = self.attention(x, active_coords, warped_canvas_coords)

        # 3. Aggregate and predict (using mean pooling over active tokens)
        logits = self.task_head(x.mean(dim=1))
        return logits

In [3]:
batch_size = 2
num_patches = 100  # Total patches in a frame (e.g., 10x10 grid)
embed_dim = 128
patch_size = 16 * 16 * 3

# Initialize Model, Optimizer, and Gatekeeper
model = VideoModel(embed_dim=embed_dim, num_heads=4)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
gatekeeper = CertifiedWarpGatekeeper(delta=32, kappa=16)

print("Starting Training Trace...\n")

Starting Training Trace...



In [7]:
# Fix the 2D RoPE frequency dimension mismatch
def apply_2d_rope_fixed(tokens, coords, head_dim):
    B, N, H, D = tokens.shape
    # Generate 16 frequencies instead of 8 by removing the step of 2
    freqs = torch.arange(0, head_dim // 2, device=tokens.device).float()
    inv_freq = 1.0 / (10000 ** (freqs / (head_dim // 2)))

    # Compute angles for X and Y coordinates
    angles_x = torch.einsum("bn,f->bnf", coords[..., 0], inv_freq)
    angles_y = torch.einsum("bn,f->bnf", coords[..., 1], inv_freq)

    # Construct rotary embeddings
    sin_x, cos_x = angles_x.sin(), angles_x.cos()
    sin_y, cos_y = angles_y.sin(), angles_y.cos()

    # Broadcast to heads
    sin_x = sin_x.unsqueeze(2).expand(-1, -1, H, -1)
    cos_x = cos_x.unsqueeze(2).expand(-1, -1, H, -1)

    # Apply to first half of feature dim (X-axis)
    tokens_x = tokens[..., :head_dim//2] * cos_x + torch.roll(tokens[..., :head_dim//2], shifts=1, dims=-1) * sin_x
    tokens_y = tokens[..., head_dim//2:]

    return torch.cat([tokens_x, tokens_y], dim=-1)

# Override the broken function in the global namespace
import sys
import __main__
__main__.apply_2d_rope = apply_2d_rope_fixed

# Reset the cached keys and values to prevent accumulation across multiple runs of this cell
if hasattr(model, 'attention'):
    model.attention.cache_k = None
    model.attention.cache_v = None

print("--- FRAME 1 (Prefill) ---")
# Raw pixel inputs (normalized to 0-255 for contrast math)
pixels_f1 = torch.rand(batch_size, num_patches, patch_size) * 255
coords_f1 = torch.rand(batch_size, num_patches, 2) * 1024 # Native (x,y) screen coords
targets = torch.randint(0, 10, (batch_size,))

# Forward Pass
logits_f1 = model(active_pixels=pixels_f1,
                  active_coords=coords_f1,
                  warped_canvas_coords=coords_f1) # Canvas = native on frame 1

loss_f1 = criterion(logits_f1, targets)
loss_f1.backward()
optimizer.step()
optimizer.zero_grad()
print(f"Frame 1 Loss: {loss_f1.item():.4f} | Processed all {num_patches} patches.")

--- FRAME 1 (Prefill) ---
Frame 1 Loss: 23.3219 | Processed all 100 patches.


In [8]:
print("\n--- FRAME 2 (Warp-Steered Cache Reuse) ---")
# Simulate a panning camera: the new frame's pixels are mostly similar to warped Frame 1
pixels_w2 = pixels_f1 # The canvas warped into Frame 2's view
# Simulate 25% of the frame changing (e.g., a moving object)
pixels_f2 = pixels_w2.clone()
pixels_f2[:, :25] += 50.0 # Inject a contrast change > delta (32) to the first 25 patches

# The camera moved, so the background canvas coordinates shifted under the new frame
warped_canvas_coords_f2 = coords_f1 + 15.0
active_coords_f2 = coords_f1[:, :25] + 15.0 # Native coords of the active patches

# 1. GATEKEEPER DECISION (Run on CPU/fast CUDA kernel before model)
keep_mask = gatekeeper.get_active_mask(pixels_f2, pixels_w2)

# Extract only the active subset for the expensive model forward pass
# For variable sizes in a batch, we pad or loop; here we take batch index 0 for clarity
active_indices = keep_mask[0].nonzero(as_tuple=True)[0]
active_pixels_f2 = pixels_f2[:, active_indices, :]
active_coords_f2 = active_coords_f2[:, active_indices, :]


--- FRAME 2 (Warp-Steered Cache Reuse) ---


In [11]:
num_active = active_indices.shape[0]
print(f"Gatekeeper dropped {num_patches - num_active} redundant patches.")
print(f"Routing {num_active} active patches to the model...")

# Ensure cache is reset first to clear any previous failed run's accumulation
model.attention.cache_k = None
model.attention.cache_v = None

# --- Frame 1 Re-run (Prefill) ---
logits_f1 = model(active_pixels=pixels_f1,
                  active_coords=coords_f1,
                  warped_canvas_coords=coords_f1)
loss_f1 = criterion(logits_f1, targets)
loss_f1.backward()
optimizer.step()
optimizer.zero_grad()

# --- Detach KV Cache ---
# Detach the cache tensors so that PyTorch doesn't backpropagate into Frame 1 through them
if model.attention.cache_k is not None:
    model.attention.cache_k = model.attention.cache_k.detach()
if model.attention.cache_v is not None:
    model.attention.cache_v = model.attention.cache_v.detach()

# --- Frame 2 (Cache Reuse) ---
# Every element in the cache needs a matching coordinate for 2D RoPE rotation.
cumulative_cache_coords = torch.cat([warped_canvas_coords_f2, active_coords_f2], dim=1)

# 2. MODEL FORWARD PASS (Uses KV Cache with matching cumulative coordinates)
logits_f2 = model(active_pixels=active_pixels_f2,
                  active_coords=active_coords_f2,
                  warped_canvas_coords=cumulative_cache_coords)

# 3. LOSS COMPUTATION & BACKWARD PASS
loss_f2 = criterion(logits_f2, targets)
loss_f2.backward()
optimizer.step()
optimizer.zero_grad()
print(f"Frame 2 Loss: {loss_f2.item():.4f} | Training step complete.")

Gatekeeper dropped 75 redundant patches.
Routing 25 active patches to the model...
Frame 2 Loss: 9.4369 | Training step complete.
